Environment

In [ ]:
libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet",
    "pycomet": "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}

In [ ]:
import sys
for lib in libs.values():
    sys.path.append(lib)

Import libraries

In [ ]:
import os
import json
import copy
import pycomet
import numpy as np
import scipy as sp
import tensorflow as tf

from scipy.linalg import expm
from silx.io.dictdump import h5todict

from hypernet import const
from pycomet.networks import FwdEulerME
from hypernet.apps import utils as hyputils
from hypernet.thermophysics import specie_thermo as st_mod

Plotting

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

path_to_figs = "./figs/"
os.makedirs(path_to_figs, exist_ok=True)

def plot_var(
    x=None,
    y=None,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    n, d = y.shape if (x is not None) else yref.shape
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=n//20,
        ls='--',
        lw=1
    )
    for i in range(d):
        if (x is not None):
            ax.plot(x, y[:,i], c=COLORS[i], **pred_style)
        if (xref is not None):
            ax.plot(xref, yref[:,i], c=COLORS[i], **true_style)
        ax.plot([], [], c=COLORS[i], label=names[i])
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.show()
    plt.close()

Initializing thermophysical models

In [ ]:
with open("./physics/inputs.json") as file:
    phyinp = json.load(file)
# Species thermo
sp_th = hyputils._get_specie_thermos(phyinp)
# Mixture
mix = hyputils._get_mixture(phyinp, sp_th)
# Kinetics
kin = hyputils._get_kinetics(phyinp, sp_th, mix)

Construct rate matrices

In [ ]:
kef, keb = 0.0, 0.0
for reaction in kin.reactions:
    if (reaction.param['process'] in (5,6)):
        kef = kef + reaction.fwd.k.squeeze()
        keb = keb + reaction.bwd.k.squeeze()
    elif (reaction.param['process'] == 2):
        kd = reaction.fwd.k.squeeze()
        kr = reaction.bwd.k.squeeze()
ke = kef + keb.T
ke = (ke.T - np.diag(np.sum(ke, axis=-1)))

Governing equations

In [ ]:
qa, qm = [st.pf.tot.Q.reshape(-1) for st in sp_th.values()]

In [ ]:
def fun(t, c):
    f = np.zeros_like(c)
    # Extract number densities
    n = c * const.UNA
    na, nm = n[0:1], n[1:]
    # Construct source terms
    we = np.matmul(ke,nm*na)
    wd = kd*nm*na - kr*na**3
    # Compose source terms
    f[0] = 2*np.sum(wd)
    f[1:] = we - wd
    return f / const.UNA

def solution(t, n0):
    res = sp.integrate.solve_ivp(
        fun=fun,
        t_span=[t[0],t[-1]],
        t_eval=t.squeeze(),
        y0=n0/const.UNA,
        method='LSODA',
        atol=1e-9,
        rtol=1e-7
    )
    return res.y.T * const.UNA

In [ ]:
ntest = solution(t, n[0])

In [ ]:
plot_var(
    x=t.squeeze()[1:],
    y=ntest[1:],
    xref=t.squeeze()[1:],
    yref=n[1:],
    labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
    scales=['log', 'log'],
    names=['O'] + [r'O$_2^{(%d)}$' % (i+1) for i in range(3)],
    figname=path_to_figs + "/solution"
)

Load master equation model

In [ ]:
model = FwdEulerME(
    processes=("exc", "dis"),
    fwd_rates={"kef": kef, "kd": kd},
    part_fn={"atom": qa, "molecule": qm}
)

In [ ]:
dnmdt_true = np.vstack([fun(t=0, c=ni/const.UNA) for ni in n])[:,1:]

In [ ]:
nt = len(t)
nb = n.shape[1]-1
x = {
    "t": np.ones((nt,1)),
    "n": n,
    "f": (np.ones((nt,nb,nb)), np.ones((nt,nb)))
}
xl = list(x.values())
dnmdt_pred = (model(xl) - n[:,1:]).numpy().squeeze() / const.UNA

In [ ]:
plot_var(
    x=t.squeeze()[1:],
    y=dnmdt_pred[1:],
    xref=t.squeeze()[1:],
    yref=dnmdt_true[1:],
    labels=[r'$t$ [s]', r'$\omega$ [m$^{-3}$s$^{-1}$]'],
    scales=['log', 'linear'],
    names=[r'O$_2^{(%d)}$' % (i+1) for i in range(dnmdt_true.shape[1])],
    figname=path_to_figs + "/source"
)

In [ ]:
import tensorflow as tf

In [ ]:
n = 1000
m = 50
a = tf.random.uniform((n,m,m))
a = tf.cast(a, dtype=tf.float64)
b = tf.random.uniform((n,m,1))
b = tf.cast(b, dtype=tf.float64)

In [ ]:
%timeit tf.einsum("nij,njk->nik", a, b) + tf.einsum("nij,njk,nkl->nil", a, a, b)

In [ ]:
%timeit tf.matmul(a, b) + tf.matmul(tf.matmul(a, a), b)

In [ ]:
%timeit tf.matmul(a, b) + tf.matmul(a, tf.matmul(a, b))

In [ ]:
y1 = tf.matmul(a, b) + tf.matmul(a, tf.matmul(a, b))
y1[0,:,0]

In [ ]:
y2 = tf.matmul(a, b) + tf.matmul(tf.matmul(a, a), b)
y2[0,:,0]

In [ ]:
tf.reduce_mean(tf.abs(y1-y2))

In [ ]:
import sys
sys.path.append("/home/zanardi/Codes/ML/PyCOMET/pycomet")

In [ ]:
import tensorflow as tf
from pycomet.operations.linalg.matrix_exp import expm

In [ ]:
n = 1000
m = 2
x = tf.random.uniform((n,m,m))
x = tf.cast(x, dtype=tf.float64)

In [ ]:
%timeit expm.expm(x, max_degree=1, max_nb_squarings=0)

In [ ]:
%timeit expm.expm(x, max_degree=2, max_nb_squarings=0)

In [ ]:
%timeit expm.expm(x, max_degree=4, max_nb_squarings=0)

In [ ]:
%timeit expm.expm(x, max_degree=8, max_nb_squarings=0)

In [ ]:
%timeit expm.expm(x, max_degree=12, max_nb_squarings=0)

In [ ]:
%timeit expm.expm(x, max_degree=18, max_nb_squarings=0)

In [ ]:
%timeit expm.expm(x, max_degree=18, max_nb_squarings=1)

In [ ]:
%timeit expm.expm(x, max_degree=18, max_nb_squarings=2)

In [ ]:
%timeit expm.expm(x, max_degree=18, max_nb_squarings=3)

In [ ]:
%timeit expm.expm(x)

In [ ]:
import sys
sys.path.append("/home/zanardi/Codes/ML/PyCOMET/pycomet")

In [ ]:
import numpy as np
import tensorflow as tf
from pycomet.operations.linalg.matrix_exp import expm, _compute_l1_norm

In [ ]:
n = 1000
m = 2

In [ ]:
norms = np.geomspace(1e-18, 1e3, 50)
for i, s in enumerate(norms):
  print("----------")
  x = tf.random.uniform((n,m,m))
  x = tf.cast(x, dtype=tf.float64)
  x = (s / _compute_l1_norm(x)) * x
  # ----------
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    y_true = tf.reduce_sum(tf.linalg.expm(x))
    y_pred = tf.reduce_sum(expm(x))
  dy_true_dx = tape.gradient(y_true, x)
  dy_pred_dx = tape.gradient(y_pred, x)
  # ----------
  err = 1e2 * tf.math.reduce_mean(tf.abs(y_true - y_pred) / y_true).numpy()
  derr = 1e2 * tf.math.reduce_mean(tf.abs(dy_true_dx - dy_pred_dx) / tf.abs(dy_true_dx)).numpy()
  print("i, s:", i, s)
  print("y_true:", tf.math.reduce_max(y_true))
  print("y_pred:", tf.math.reduce_max(y_pred))
  print("dy_true_dx:", tf.math.reduce_max(dy_true_dx))
  print("dy_pred_dx:", tf.math.reduce_max(dy_pred_dx))
  print("err:", err)
  print("derr:", derr)
  print("----------")

In [ ]:
x = tf.constant(tf.random.uniform((2,2)))

In [ ]:
for xi in x:
    print(xi[0]*x)

In [ ]:
nb_threads = 2
cpu = tf.config.list_physical_devices("CPU")[0]
tf.config.set_visible_devices(cpu)
tf.config.threading.set_inter_op_parallelism_threads(nb_threads)
tf.keras.backend.set_floatx('float64')
tf.config.threading.set_intra_op_parallelism_threads(nb_threads)

In [ ]:
def eigen(x):
  l, v = tf.linalg.eig(x)
  vinv = tf.linalg.inv(v)

In [ ]:
def compute_v1(x):
    l, v = tf.linalg.eig(x)
    vinv = tf.linalg.inv(v)
    expx = tf.einsum("...ij,...j,...jk->...ik", v, tf.math.exp(l), vinv)
    xinv = tf.einsum("...ij,...j,...jk->...ik", v, 1/l, vinv)
    return [tf.math.real(z) for z in (expx, xinv)]

def compute_v2(x):
    expx = tf.linalg.expm(x)
    xinv = tf.linalg.inv(x)
    return expx, xinv

In [ ]:
x = tf.random.uniform((10,10))
x = tf.cast(x, dtype="float64")

In [ ]:
res1 = compute_v1(x)
res2 = compute_v2(x)

In [ ]:
for i in range(len(res1)):
    print(tf.reduce_mean(tf.abs(res1[i] - res2[i])))

In [ ]:
n = 1000
m = 100

In [ ]:
x = tf.random.uniform((n,m,m))
x = tf.cast(x, dtype="float64")

In [ ]:
tf.linalg.inv(x) - tf.linalg.pinv(x)

In [ ]:
tf.linalg.pinv(x)

In [ ]:
%timeit tf.linalg.svd(x)

In [ ]:
%timeit tf.linalg.lu(x)

In [ ]:
%timeit tf.linalg.eig(x)

In [ ]:
%timeit tf.linalg.inv(x)

In [ ]:
%timeit tf.matmul(x, x)

In [ ]:
import scipy as sp
import numpy as np

In [ ]:
x = np.random.rand(5,5)
T, V = sp.linalg.schur(x)

In [ ]:
import scipy.linalg
import numpy.random
import numpy

# Setup
N = 100
H = 0.01*numpy.random.rand(N,N)
H = H+H.T # symmetric
S = numpy.random.rand(N,N)
S = S+S.T # symmetric
S = S.dot(S) # positive definite
Sinv = scipy.linalg.inv(S)
I = numpy.identity(N)

# Reference answer
ref = scipy.linalg.expm(Sinv.dot(H))

# Our solution
## first our cholesky factors
L = scipy.linalg.cholesky(S,lower=True)
Linv = scipy.linalg.solve_triangular(L, I, lower=True)

## now solve the normal eigenvalue problem
Hprime = Linv.dot(H).dot(Linv.T)
D, Xprime = scipy.linalg.eigh(Hprime)
D = numpy.diag(D)

## back subsite for X and X^{-1}
X = Linv.T.dot(Xprime)
Xinv = Xprime.T.dot(L.T)

our_answer = X.dot(scipy.linalg.expm(D)).dot(Xinv)

print(scipy.linalg.norm(ref - our_answer))

In [ ]:
A = Sinv.dot(H)
A.shape

In [ ]:
%timeit scipy.linalg.expm(A)